[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/ttl/time-to-live.ipynb)

# Time-to-Live (TTL) in Weaviate

Weaviate **v1.36+** can automatically expire objects in a collection. You configure a TTL
policy on the collection and a background process periodically deletes objects once they
have lived past their expiration.

There are **three TTL modes**, and this recipe demonstrates all of them with the public
[HuffPost News Category dataset](https://huggingface.co/datasets/heegyu/news-category-dataset):

| Mode | Constructor | Object expires at | Use case in this recipe |
|------|-------------|-------------------|--------------------------|
| Creation time | `Configure.ObjectTTL.delete_by_creation_time` | `created + time_to_live` | **Ephemeral cache / session store** |
| Update time | `Configure.ObjectTTL.delete_by_update_time` | `last_updated + time_to_live` | **Keep-alive / inactivity-based retention** |
| Date property | `Configure.ObjectTTL.delete_by_date_property` | `value_of(property) + ttl_offset` | **News freshness window** |

### Requirements & notes

- **Weaviate v1.36+**.
- The background sweeper runs on a cron schedule defined by the **`OBJECTS_TTL_DELETE_SCHEDULE`**
  environment variable. Without it, expired objects are *not* deleted. For this demo we use
  `"* * * * *"` (every minute) so we can watch objects disappear quickly.
- **Minimum TTL is 60 seconds.**
- `filter_expired_objects=True` excludes expired-but-not-yet-swept objects from query results,
  so reads stay correct in the gap between expiry and the next background sweep.

> Because the minimum TTL is 60s and the sweeper runs once per minute, **each section below
> includes a real wait of ~1–2.5 minutes** while objects expire. The full notebook takes a
> few minutes to run end-to-end.


In [ ]:
!pip install -Uq weaviate-client datasets

## 1. Connect to an embedded Weaviate

We enable the TTL background sweeper by setting `OBJECTS_TTL_DELETE_SCHEDULE` to a cron expression that fires every minute.

In [2]:
import weaviate

client = weaviate.connect_to_local()

print("Weaviate ready:", client.is_ready())

Weaviate ready: True


## 2. Load a public dataset

We stream a small sample of the HuffPost News Category dataset (no auth token required). We only need the text fields — the real `date` field is not used directly (Section C assigns its own dates so the expiry window is observable in a short demo).

In [3]:
from itertools import islice
from datasets import load_dataset

# stream so we don't download the full 87MB file
stream = load_dataset("heegyu/news-category-dataset", split="train", streaming=True)

SAMPLE_SIZE = 150
articles = [
    {
        "headline": row["headline"],
        "category": row["category"],
        "short_description": row["short_description"],
    }
    for row in islice(stream, SAMPLE_SIZE)
]

print(f"Loaded {len(articles)} articles")
articles[0]

/Users/dudanogueira/dev/weaviate/recipes/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 150 articles


{'headline': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters',
 'category': 'U.S. NEWS',
 'short_description': 'Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.'}

## 3. Helpers

A `count()` helper (optionally filtered) and a `watch()` poller that prints live counts so we can see objects vanish.

In [4]:
import time
from weaviate.classes.query import Filter


def count(collection, where=None):
    """Total objects in the collection, optionally matching a filter."""
    return collection.aggregate.over_all(filters=where).total_count


def watch(collection, timeout=180, interval=15, groups=None, done=None):
    """Poll the collection count every `interval`s until `done` or `timeout`.

    `groups` is an optional dict of {label: Filter} to also print per-group counts.
    `done` is an optional predicate `done(collection) -> bool`; polling stops as soon as
    it returns True (default: stop when the collection is empty). This lets the cell
    finish automatically once expiry is complete, instead of always waiting the full
    timeout (no need to interrupt the kernel manually).
    """
    groups = groups or {}
    done = done or (lambda c: count(c) == 0)
    start = time.time()
    while True:
        elapsed = int(time.time() - start)
        parts = [f"total={count(collection)}"]
        for label, f in groups.items():
            parts.append(f"{label}={count(collection, f)}")
        print(f"[{elapsed:>4}s]  " + "   ".join(parts))
        if done(collection):
            print("✅ done — expired objects have been swept")
            break
        if time.time() - start >= timeout:
            print("⏱️  timeout reached (increase `timeout` if needed)")
            break
        time.sleep(interval)

## 4. Section A — Creation-time TTL (ephemeral cache / session store)

`delete_by_creation_time` expires every object a fixed duration after it was inserted.
This is ideal for **ephemeral data**: session context for an AI agent, cached results,
one-time tokens — anything that should simply disappear after a short window.

We set a 60-second TTL (the minimum) and `filter_expired_objects=True` so queries never
return stale objects, even before the background sweep runs.

In [5]:
from weaviate.classes.config import Configure

client.collections.delete("NewsEphemeral")
ephemeral = client.collections.create(
    name="NewsEphemeral",
    object_ttl_config=Configure.ObjectTTL.delete_by_creation_time(
        time_to_live=60,            # seconds (minimum allowed)
        filter_expired_objects=True,
    ),
)

with ephemeral.batch.fixed_size(50) as batch:
    for a in articles:
        batch.add_object(a)

print("Inserted:", count(ephemeral))

Inserted: 150


In [6]:
# BM25 keyword search works right after insert...
res = ephemeral.query.bm25(query="election", limit=3)
for o in res.objects:
    print("-", o.properties["headline"])

- Michigan Secretary of State Worried About ‘Violence And Disruption’ Going Into Midterms
- Democrats Nominate Seth Magaziner In Key Rhode Island House Race


Now we wait. Every object was created at roughly the same moment, so they all expire ~60s later and the next sweep removes them. The count should fall to **0**.

In [7]:
# objects expire 60s after creation; sweeper runs every minute.
# watch() stops automatically once the collection is empty (default `done`).
watch(ephemeral, timeout=180, interval=15)
print("\nFinal count:", count(ephemeral))

[   0s]  total=150
[  15s]  total=150
[  30s]  total=150
[  45s]  total=150
[  60s]  total=150
[  75s]  total=0
✅ done — expired objects have been swept

Final count: 0


## 5. Section B — Update-time TTL (keep-alive / inactivity-based retention)

`delete_by_update_time` resets the expiry clock every time an object is **updated**.
Objects that keep being touched survive; objects that go untouched expire. This models
**inactivity-based retention** — e.g. keep active user/session records, automatically purge
stale ones (a common data-retention / "right to be forgotten by inactivity" pattern).

We tag half the objects with `kept_alive=True`. During the wait we periodically *update* only
those, refreshing their clock, while the untouched half expires.

In [8]:
client.collections.delete("NewsKeepAlive")
keepalive = client.collections.create(
    name="NewsKeepAlive",
    object_ttl_config=Configure.ObjectTTL.delete_by_update_time(
        time_to_live=60,            # seconds
        filter_expired_objects=True,
    ),
)

with keepalive.batch.fixed_size(50) as batch:
    for i, a in enumerate(articles):
        batch.add_object({**a, "kept_alive": i % 2 == 0, "refreshes": 0})

kept = Filter.by_property("kept_alive").equal(True)
stale = Filter.by_property("kept_alive").equal(False)

# remember the UUIDs we will keep refreshing
kept_uuids = [o.uuid for o in keepalive.query.fetch_objects(filters=kept, limit=1000).objects]

print("total:", count(keepalive), "| kept_alive:", count(keepalive, kept), "| untouched:", count(keepalive, stale))

total: 150 | kept_alive: 75 | untouched: 75


We now run for ~2.5 minutes. Every 20s we update the `kept_alive` objects to reset
their clock, while leaving the others untouched. Watch `kept` stay constant while `stale`
drops to **0**.

> **⚠️ Important:** the update must **change a value**. A *no-op* update that writes the same
> property values back does **not** bump the object's update timestamp, so the TTL clock is
> **not** reset and the object expires anyway. Here we increment a `refreshes` counter so each
> update is a real change. (We also wrap the call in `try/except`: an object that has already
> expired and been swept returns a `404`.)

In [9]:
n = 0
start = time.time()
while time.time() - start < 150:
    n += 1
    # refresh only the kept-alive objects -> resets their update-time TTL clock.
    # NOTE: we must change a value (the `refreshes` counter); a no-op update would
    # NOT reset the clock, and the object would expire anyway.
    for u in kept_uuids:
        try:
            keepalive.data.update(uuid=u, properties={"refreshes": n})
        except Exception:
            pass  # object may have already expired and been swept (404)

    elapsed = int(time.time() - start)
    print(f"[{elapsed:>4}s]  total={count(keepalive)}   kept={count(keepalive, kept)}   stale={count(keepalive, stale)}")

    # stop once the untouched half is gone (the kept-alive half has survived)
    if count(keepalive, stale) == 0:
        print("✅ done — untouched objects expired; kept-alive objects survived")
        break
    time.sleep(20)

print("\nFinal -> kept:", count(keepalive, kept), "| untouched:", count(keepalive, stale))

[   0s]  total=150   kept=75   stale=75
[  20s]  total=150   kept=75   stale=75
[  40s]  total=150   kept=75   stale=75
[  60s]  total=150   kept=75   stale=75
[  80s]  total=150   kept=75   stale=75
[ 101s]  total=150   kept=75   stale=75
[ 121s]  total=75   kept=75   stale=0
✅ done — untouched objects expired; kept-alive objects survived

Final -> kept: 75 | untouched: 0


## 6. Section C — Date-property TTL (news freshness window)

`delete_by_date_property` expires an object based on one of its own **DATE properties**:
the object expires at `value_of(property) + ttl_offset` (the offset may even be negative).
This is perfect for a **freshness window** — keep recent news searchable and let old news
age out automatically.

The real articles are from 2022, so we assign each one a synthetic `publishedDate` purely so
the window is observable in a 60-second demo: half are "fresh" (published *now*) and half are
"old" (published an hour ago). With a 5-minute offset, the old ones are already past expiry
and get swept on the next run, while the fresh ones survive.

In [10]:
from datetime import datetime, timezone, timedelta
from weaviate.classes.config import Property, DataType

client.collections.delete("NewsFreshness")
freshness = client.collections.create(
    name="NewsFreshness",
    properties=[
        Property(name="headline", data_type=DataType.TEXT),
        Property(name="category", data_type=DataType.TEXT),
        Property(name="freshness", data_type=DataType.TEXT),
        Property(name="publishedDate", data_type=DataType.DATE),
    ],
    object_ttl_config=Configure.ObjectTTL.delete_by_date_property(
        property_name="publishedDate",
        ttl_offset=timedelta(minutes=5),   # expire 5 min after publishedDate
        filter_expired_objects=True,
    ),
)

now = datetime.now(timezone.utc)
with freshness.batch.fixed_size(50) as batch:
    for i, a in enumerate(articles):
        is_fresh = i % 2 == 0
        published = now if is_fresh else now - timedelta(hours=1)  # old -> already past expiry
        batch.add_object({
            "headline": a["headline"],
            "category": a["category"],
            "freshness": "fresh" if is_fresh else "old",
            "publishedDate": published,   # tz-aware datetime (Weaviate DATE = RFC3339)
        })

fresh = Filter.by_property("freshness").equal("fresh")
old = Filter.by_property("freshness").equal("old")
print("total:", count(freshness), "| fresh:", count(freshness, fresh), "| old:", count(freshness, old))

total: 150 | fresh: 75 | old: 75


The "old" articles (publishedDate + 5min is already in the past) get swept on the next run; the "fresh" ones remain. Watch `old` drop to **0** while `fresh` stays.

In [11]:
# stop as soon as the "old" articles are gone (the "fresh" ones persist)
watch(freshness, timeout=180, interval=15, groups={"fresh": fresh, "old": old},
      done=lambda c: count(c, old) == 0)
print("\nFinal -> fresh:", count(freshness, fresh), "| old:", count(freshness, old))

[   0s]  total=150   fresh=75   old=75
[  15s]  total=150   fresh=75   old=75
[  30s]  total=150   fresh=75   old=75
[  45s]  total=75   fresh=75   old=0
✅ done — expired objects have been swept

Final -> fresh: 75 | old: 0


## 7. Cleanup

In [12]:
client.collections.delete(["NewsEphemeral", "NewsKeepAlive", "NewsFreshness"])
client.close()
print("Done.")

Done.
